# Fase 2i: CT mixed-context con slices negativos

Este experimento prueba una mejora de datos: entrenar con slices positivos y una muestra controlada de slices negativos del mismo conjunto de estudios de train. La evaluacion principal se mantiene en los slices positivos de val/test para ser comparable con los resultados anteriores.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import pandas as pd

from src.config import config
from src.data.segmentation import build_ct_segmentation_dataframe, split_segmentation_dataframe
from src.training.segmentation_experiment import (
    get_device,
    make_segmentation_run_config,
    train_and_evaluate_segmentation,
    save_segmentation_artifacts,
    existing_segmentation_artifact,
    summarize_segmentation_results,
)

config.NUM_WORKERS = 0
device = get_device()
print(f'Device: {device}')

## Configuracion

`NEGATIVE_RATIO = 1.0` significa que train usa aproximadamente un slice negativo por cada slice positivo. Si empeora demasiado por conservador, prueba despues `0.5`. Esta version usa `base_features=32` para compararse con el nuevo mejor modelo CT.

In [ ]:
NEGATIVE_RATIO = 1.0

run_config = make_segmentation_run_config(
    dataset_name='ct',
    architecture='attention_unet',
    run_mode='full',
    image_size=config.CT_IMAGE_SIZE,
    in_channels=1,
    variant_name='mixed30_patch192_pos70_tversky_pos10_bf32_neg1_thr095',
    loss_name='weighted_tversky_bce',
    bce_weight=0.2,
    dice_weight=0.8,
    pos_weight=10.0,
    tversky_alpha=0.3,
    tversky_beta=0.7,
    optimize_threshold=True,
    threshold_search_max=0.95,
    train_crop_size=(192, 192),
    train_crop_prob=0.3,
    positive_crop_prob=0.7,
    batch_size=4,
    epochs=36,
    early_stopping_patience=8,
    base_features=32,
)
run_config

## Datos

Se usan los mismos estudios de train/val/test que en los experimentos positivos. Solo se amplian los slices de train con negativos.

In [ ]:
positive_df = build_ct_segmentation_dataframe(
    config.CT_DIR,
    config.CT_DIR / 'processed_segmentation_slices',
    target_size=config.CT_IMAGE_SIZE,
    positive_mask_only=True,
    overwrite=False,
)

train_pos_df, val_df, test_df = split_segmentation_dataframe(
    positive_df,
    random_seed=config.RANDOM_SEED,
    group_col='study_id',
)

all_df = build_ct_segmentation_dataframe(
    config.CT_DIR,
    config.CT_DIR / 'processed_segmentation_slices_all',
    target_size=config.CT_IMAGE_SIZE,
    positive_mask_only=False,
    overwrite=False,
)
all_df['has_mask'] = all_df['has_mask'].astype(str).str.lower().isin(['true', '1'])


train_all_df = all_df[all_df['study_id'].isin(train_pos_df['study_id'].unique())].copy()
train_positive = train_all_df[train_all_df['has_mask'].astype(bool)].copy()
train_negative = train_all_df[~train_all_df['has_mask'].astype(bool)].copy()
n_negative = min(len(train_negative), int(len(train_positive) * NEGATIVE_RATIO))
train_negative = train_negative.sample(n=n_negative, random_state=config.RANDOM_SEED) if n_negative else train_negative.head(0)
train_df = pd.concat([train_positive, train_negative], ignore_index=True).sample(frac=1.0, random_state=config.RANDOM_SEED).reset_index(drop=True)

pd.DataFrame([
    {'split': 'train', 'slices': len(train_df), 'positive': int(train_df['has_mask'].sum()), 'negative': int((~train_df['has_mask'].astype(bool)).sum()), 'studies': train_df.study_id.nunique()},
    {'split': 'val_positive_only', 'slices': len(val_df), 'positive': len(val_df), 'negative': 0, 'studies': val_df.study_id.nunique()},
    {'split': 'test_positive_only', 'slices': len(test_df), 'positive': len(test_df), 'negative': 0, 'studies': test_df.study_id.nunique()},
])

## Entrenamiento

In [ ]:
seg_models_dir = config.MODELS_DIR / 'segmentation' / 'ct'
seg_results_dir = PROJECT_ROOT / 'results' / 'segmentation' / 'ct'

if existing_segmentation_artifact(run_config, seg_models_dir, seg_results_dir):
    print(f'Saltado: artefactos existentes detectados para {run_config.experiment_name}_{run_config.run_mode}.')
else:
    result = train_and_evaluate_segmentation(run_config, train_df, val_df, test_df, device)
    artifact_paths = save_segmentation_artifacts(run_config, result, seg_models_dir, seg_results_dir)
    print(artifact_paths)
    print(result['metrics'])
    print('threshold:', result['threshold'])

## Comparacion

In [ ]:
summary_df = summarize_segmentation_results(sorted(seg_results_dir.glob('*_summary.json')))
cols = ['experiment', 'variant_name', 'loss_name', 'dice', 'iou', 'pixel_accuracy', 'threshold', 'hyperparameters']
summary_df[cols].sort_values('dice', ascending=False).reset_index(drop=True)